## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e3

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E3" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E3/smoke" if SMOKE else "E3"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e3.build_initial_spec(SMOKE)

## 2. Write spec

In [2]:
report.write_spec(SPEC, OUT, modules=e3.E3_MODULES)
e3.verify_spec_immutable(OUT, SPEC, e3.E3_MODULES)

## 3. Leakage test

In [3]:
leaks = pd.DataFrame(e3.run_leakage_e3(SPEC), columns=["pipeline", "pass", "note"])
report.save_table(leaks, OUT, "leakage")
assert leaks["pass"].all(), leaks[~leaks["pass"]]

condpred.rp_batch: backend=torch device=cuda (NVIDIA GeForce RTX 4080 Laptop GPU) torch=2.10.0+cu128


## 4. Simulate clean paths (cached)

In [4]:
e3.simulate_all(SPEC, CACHE)

## 5. Contaminate (cached per seed, all conditions)

In [5]:
e3.contaminate_all(SPEC, CACHE)

## 6. E3a fidelity

In [6]:
POOL = e3.make_pool()
E3A = e3.run_e3a(SPEC, OUT, CACHE, POOL)
display(E3A["fidelity"].query("system == 'ms_sv' and W == 120 and cond in ['J_l0.2_k5', 'N_w0.05']"))

E3 pool: 16 workers


,system,cond,factor,level,kind,arm,input,W,jaccard_mean,jaccard_sd,n_seeds
472,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,BV-wz,logbv,120,0.748653,0.022454,30
473,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-beta(0),rv,120,0.531882,0.028358,30
474,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-beta(0.5),rv,120,0.498709,0.030777,30
476,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-beta(1),rv,120,0.486059,0.032010,30
478,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-beta(3),rv,120,0.493531,0.033326,30
479,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-beta(5),rv,120,0.502140,0.033978,30
480,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-betaexp(0),ret,120,0.539897,0.027297,30
481,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-betaexp(1),ret,120,0.548142,0.027644,30
482,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-betaexp(3),ret,120,0.586036,0.031007,30
483,ms_sv,J_l0.2_k5,jumps,0.2000,arm_clean_vs_contaminated,G-betaexp(5),ret,120,0.610535,0.033654,30


## 7. E3b regime detection

In [7]:
E3B = e3.run_e3b(SPEC, OUT, CACHE)
display(E3B["auc"].query("train_on == 'same' and W == 120").pivot_table(index="model", columns="cond", values="auc_pooled"))

E3b: 2166 logistic fits


cond,C_JN_normal,C_JN_t5,C_J_t5,C_N_t5,J_l0.05_k3,J_l0.05_k5,J_l0.05_k8,J_l0.2_k3,J_l0.2_k5,J_l0.2_k8,J_l0.5_k3,J_l0.5_k5,J_l0.5_k8,M_block_0.1,M_block_0.2,M_block_0.4,M_block_0.6,M_random_0.1,M_random_0.2,M_random_0.4,M_random_0.6,N_w0.02,N_w0.05,N_w0.1,S_0.05,S_0.1,S_0.2,T_t5,clean
model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
B-PE,0.582008,0.565722,0.518564,0.531515,0.527326,0.519068,0.486819,0.467576,0.510996,0.501422,0.495449,0.541065,0.513339,0.548193,0.473538,0.486253,0.504609,0.527361,0.465985,0.502116,0.469774,0.491154,0.595055,0.572895,0.487121,0.503179,0.489788,0.526979,0.528093
B-rollBV,0.715623,0.716023,0.707570,0.717193,0.707597,0.707441,0.707183,0.707569,0.708340,0.708926,0.708783,0.707446,0.707893,0.695808,0.694909,0.683046,0.685154,0.704972,0.706051,0.706573,0.690146,0.709217,0.715332,0.716702,0.706322,0.706374,0.703498,0.707661,0.707477
B-rollRV,0.714357,0.717825,0.708151,0.716625,0.708550,0.708668,0.707374,0.708107,0.709569,0.711631,0.711483,0.710977,0.706563,0.696893,0.697261,0.681677,0.686285,0.705831,0.706813,0.707194,0.689999,0.709826,0.715695,0.717224,0.707041,0.707237,0.704747,0.708753,0.708207
BV-wz,0.758095,0.763739,0.691293,0.779347,0.672339,0.676567,0.673457,0.682716,0.677296,0.679507,0.671641,0.714092,0.696442,0.687999,0.655339,0.613433,0.585980,0.681320,0.659076,0.669984,0.589194,0.692307,0.770064,0.802592,0.673152,0.684709,0.675480,0.682599,0.677202
G-beta(0),0.740530,0.728424,0.684114,0.765639,0.654446,0.665331,0.677023,0.691297,0.698746,0.707421,0.711219,0.730170,0.756451,0.679700,0.651116,0.592038,0.566598,0.691420,0.670123,0.678159,0.582455,0.695510,0.766384,0.795458,0.687696,0.662066,0.698477,0.679539,0.675525
G-beta(0.5),0.795614,0.771446,0.725733,0.775074,0.753348,0.742076,0.744659,0.747291,0.731558,0.724825,0.751369,0.704287,0.708822,0.746636,0.690253,0.637220,0.606439,0.765720,0.759706,0.753580,0.642101,0.775703,0.802974,0.798950,0.770552,0.754809,0.772495,0.759599,0.766671
G-beta(1),0.791418,0.773642,0.748754,0.784854,0.767112,0.752039,0.763222,0.779078,0.766573,0.754632,0.786638,0.767941,0.757910,0.763353,0.729244,0.665711,0.642929,0.776086,0.772834,0.767225,0.678618,0.775978,0.797358,0.797637,0.786398,0.768981,0.786345,0.771176,0.771529
G-beta(3),0.790871,0.781364,0.763657,0.782584,0.774204,0.781872,0.765478,0.800938,0.768272,0.737128,0.788710,0.788894,0.764494,0.774262,0.749605,0.672264,0.686980,0.776796,0.768807,0.760847,0.705856,0.785360,0.788320,0.780578,0.780961,0.777285,0.767137,0.766131,0.783140
G-beta(5),0.783564,0.780039,0.752229,0.792926,0.763178,0.742981,0.728043,0.771308,0.747890,0.710505,0.779323,0.772162,0.731347,0.740285,0.702139,0.652228,0.677229,0.758132,0.751104,0.741486,0.705515,0.763548,0.796607,0.773653,0.741424,0.748786,0.757715,0.753998,0.749653


## 8. E3d missing and stale data

In [8]:
E3D = e3.run_e3d(SPEC, OUT, CACHE, POOL)
POOL.shutdown()
display(E3D["table"][["series", "pattern", "method", "rel_error_mean", "jaccard_mean", "drift_DET_mean", "drift_LAM_mean"]])

,series,pattern,method,rel_error_mean,jaccard_mean,drift_DET_mean,drift_LAM_mean
0,logRV,M_block_0.1,ffill,0.151755,0.680207,0.518623,0.540274
1,logRV,M_block_0.1,hankel,0.120862,0.698555,0.440615,0.479392
2,logRV,M_block_0.1,linear,0.120036,0.702220,0.449021,0.503988
3,logRV,M_block_0.2,ffill,0.249638,0.469805,0.930609,1.055033
4,logRV,M_block_0.2,hankel,0.195221,0.499767,0.797088,0.912479
5,logRV,M_block_0.2,linear,0.194723,0.503418,0.798610,0.946508
6,logRV,M_block_0.4,ffill,0.393707,0.237170,1.942766,1.979650
7,logRV,M_block_0.4,hankel,0.308281,0.271050,1.549924,1.662964
8,logRV,M_block_0.4,linear,0.306486,0.274772,1.562183,1.721155
9,logRV,M_block_0.6,ffill,0.491600,0.138491,3.150731,2.729367


## 9. Hypothesis verdicts

In [9]:
rows = e3.evaluate_e3a(SPEC, E3A) + e3.evaluate_e3b(SPEC, E3B) + e3.evaluate_e3d(SPEC, E3D) + [e3.e3c_row(SPEC)]
report.write_decision(rows, OUT)
display(pd.read_csv(OUT / "decision.csv"))

,gate,statistic,value,ci_low,ci_high,rule,verdict,note,smoke
0,H1a,max_b mean(J_beta(b) - J_wz) at b=0,-0.000027,-0.000091,0.000035,supported iff mean(J_beta(b) - J_wz) <= 0.02 f...,supported,"b=0: -0.0000 [-0.0001, +0.0000]; b=0.5: -0.033...",False
1,H1b,J_BV-wz - J_G-betaexp(5),0.138118,0.127493,0.147766,"supported iff CI low of J_BV-wz - J_best > 0, ...",supported,J_BV-wz=0.7487; best other arm G-betaexp(5) J=...,False
2,H2,J_beta(5) - J_beta(0),0.169135,0.161843,0.176799,supported iff mean diff >= 0.05 and CI low > 0,supported,NaN,False
3,H3,J_eps - J_wz,-0.011751,-0.022978,-0.001823,supported iff CI low of J_eps - J_wz > 0,not_supported,NaN,False
4,H4,"min_(b<=3) mean J(beta(b), pow(b)) at b=0",0.995136,0.995010,0.995261,"supported iff mean >= 0.9 for b in {0, 0.5, 1,...",not_supported,"b=0: 0.9951 [0.9950, 0.9953]; b=0.5: 0.9959 [0...",False
5,H5,loss_rollRV - loss_G-beta(3),0.001581,-0.017214,0.024745,best arm = highest clean AUC; supported iff CI...,not_supported,AUC rollRV clean 0.7082 -> 0.7144; G-beta(3) c...,False
6,E3b_jump_vs_BV,max AUC_arm - AUC_rollBV (G-beta(3)),0.059932,-0.019104,0.141742,yes iff any arm has AUC_arm - AUC_rollBV > 0 w...,no,AUC rollBV 0.7083; G-wz: -0.0097 (p_holm 1); G...,False
7,H6_logRV,"err_linear - err_hankel (M_random_0.4, lowest CI)",-0.000492,-0.001178,0.000217,supported iff CI low of (err_linear - err_hank...,not_supported,"M_random_0.4: -0.0005 [-0.0012, +0.0002]; M_ra...",False
8,H7_logRV,"drift_linear - drift_hankel (M_random_0.4 DET,...",0.070640,0.052468,0.089176,supported iff CI low of (drift_linear - drift_...,supported,"M_random_0.4 DET: +0.0706 [+0.0525, +0.0892]; ...",False
9,H6_lorenz,"err_linear - err_hankel (M_random_0.4, lowest CI)",0.044504,0.042468,0.046436,supported iff CI low of (err_linear - err_hank...,supported,"M_random_0.4: +0.0445 [+0.0425, +0.0464]; M_ra...",False


## 10. Figures and tables

In [10]:
e3.make_figures(OUT, FIG)
for name in ["fidelity", "feature_drift", "regime_auc", "missing_fill", "decision"]:
    assert (OUT / f"{name}.csv").exists(), name
print(sorted(p.name for p in FIG.glob("*.png")))

['e3d_rel_error.png', 'fidelity_lorenz.png', 'fidelity_ms_sv.png', 'fidelity_msar.png', 'regime_auc_heatmap.png']
